# Capstone G6: Estrogen, cognition, and whose symptoms get believed

Older women (60+) from a national health survey: does hormone/estrogen use relate to measured cognition? A small dataset with a big trap -- and a question about what tests actually capture.

**Your group's priority: confounding vs causation, and the patient's own account.** Women who used estrogen are also, on average, healthier, wealthier, and better-educated. So a raw 'estrogen users score better' can be entirely those other things. And a digit-symbol test never measures the 'brain fog' a patient actually reports.

This is a **tabular** problem -- a table of numbers per person, like Day 2. The model is easy;
the interesting questions are about *which features* and *who the model works for*. **Rubric:**
build it, evaluate honestly, find a failure mode, defend a decision, both partners can explain.

In [ ]:
# Setup: on Colab this grabs the course files + installs what's missing. Locally it's a no-op.
import os, sys
if not os.path.exists("../../capstone_common.py"):
    os.system("git clone -q https://github.com/jinchiwei/outset-ai-healthcare.git")
    os.chdir("outset-ai-healthcare/notebooks/day3_capstone/projects/g6_estrogen")
sys.path.insert(0, "../..")            # day3_capstone (capstone_common / _tabular / _seq)
sys.path.insert(0, "../../../_shared") # colab_setup
import colab_setup
colab_setup.ensure(*colab_setup.DAY3_TABULAR)

In [ ]:
import sys
sys.path.insert(0, "../..")
import capstone_tabular as ct

df, meta = ct.load_estrogen()
print(meta["name"], "  ->", df.shape[0], "rows")
print("features:", meta["features"])
print("predicting:", meta["target"], f"(1 = {meta['positive']})")
df.head()

## Look at the data
How many people are in each class? Is it balanced? What might be confounded?

In [ ]:
import matplotlib.pyplot as plt
print(df[meta["target"]].value_counts().rename({0: "no", 1: "yes"}).to_string())
df[meta["features"]].describe().T[["mean", "min", "max"]]

## Build your own model (interactive)

Pick which **features** the model sees and which **model** to use, hit Run, read the TEST
accuracy. Try dropping features: does accuracy hold with fewer? Which features actually matter?

In [ ]:
from ipywidgets import interact_manual, SelectMultiple, Dropdown

def build(features, model):
    if not features:
        print("Pick at least one feature."); return
    ct.fit_eval(df, list(features), meta["target"], model=model)

try:
    interact_manual(build,
        features=SelectMultiple(options=meta["features"], value=tuple(meta["features"]),
                                description="features", rows=min(10, len(meta["features"])),
                                style={"description_width": "initial"}),
        model=Dropdown(options=list(ct.make_models()), value="Logistic Regression", description="model"))
except ImportError:
    ct.fit_eval(df, meta["features"], meta["target"])

## Fairness: does the model work equally well for everyone?

Your priority is **confounding vs causation, and the patient's own account**. Train once, then measure TEST accuracy **separately for
each group** (estrogen users vs non-users). A model can look great overall and quietly fail one group.

In [ ]:
from ipywidgets import interact_manual, Dropdown

def fairness(model):
    ct.audit_by_group(df, meta["features"], meta["target"],
                      meta["group"], meta.get("group_names"), model=model)

try:
    interact_manual(fairness,
        model=Dropdown(options=list(ct.make_models()), value="Logistic Regression", description="model"))
except ImportError:
    ct.audit_by_group(df, meta["features"], meta["target"], meta["group"], meta.get("group_names"))

## The confounding trap (the heart of this project)

An association is not a cause. Below, measure the effect of one feature on the outcome **alone**,
then **again after controlling** for the obvious confounders. If the effect shrinks toward zero,
the raw association was mostly the confounders -- not the thing itself.

This is the whole question for your group: does the exposure *cause* the outcome, or do the two
just travel together because of age, education, and income?

In [ ]:
from ipywidgets import interact_manual, Dropdown, SelectMultiple

others = [f for f in meta["features"] if f != "used_estrogen"]

def confound(feature, controls):
    ct.association(df, feature, meta["target"], list(controls))

try:
    interact_manual(confound,
        feature=Dropdown(options=meta["features"], value="used_estrogen", description="effect of"),
        controls=SelectMultiple(options=others, value=('age', 'education', 'income_ratio'),
                                description="control for", rows=min(6, len(others)),
                                style={"description_width": "initial"}))
except ImportError:
    ct.association(df, "used_estrogen", meta["target"], list(('age', 'education', 'income_ratio')))

## Where to take it (with Claude)
- After controlling for age/education/income, how much of the 'estrogen helps cognition' effect is left? Write the honest one-sentence conclusion.
- The outcome is an *objective* digit-symbol test. Brain fog is *subjective*. Argue why the gap between the two is itself the story a patient would tell.
- Reflect: how does it change care when a woman's symptoms are dismissed because a test 'looks normal'? Tie your numbers to that.